# D4-v1: IndiGo next-quarter PAT margin modelling

This notebook re-runs the fixed experiment and inspects the run log, holdout predictions and model-selection outcome. The split windows are defined in src/d4_train.py and must not be changed after looking at test results.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / "src" / "d4_train.py").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.d4_train import run_experiment, SPLIT_WINDOWS, FEATURES
INPUT = ROOT / "data" / "processed" / "Final_Quarterly_Data(1).xlsx"
OUTPUT = ROOT / "milestones" / "M4_D4_Modelling" / "experiments"
metrics = run_experiment(INPUT, OUTPUT)
metrics["sample_counts"], metrics["selected_algorithm"]


## Declared time-aware split

Train: 2017Q4-2022Q4. Validation: 2023Q1-2024Q2. Test: 2024Q3-2026Q2. The validation window is used for model and hyperparameter selection. The test window is held back until selection is locked.

In [ ]:
pd.DataFrame([
    {"split": name, "start": dates[0], "end": dates[1], "n": metrics["sample_counts"][name]}
    for name, dates in SPLIT_WINDOWS.items()
])

## Validation candidate results

Baseline is persistence: predict the next PAT margin as the immediately preceding reported quarter margin. Candidates are Ridge Regression and Random Forest. Primary tuning metric is validation MAE; RMSE is the tie-breaker.

In [ ]:
results = metrics["candidate_validation"]
candidate_table = pd.DataFrame(results)
candidate_table.sort_values(["mae", "rmse"]).reset_index(drop=True)

## Experiment log and test predictions

Every model configuration is recorded in the CSV log with date, source-data hash, features, parameters, seed and metric. Only the selected model is evaluated on the final test period.

In [ ]:
log = pd.read_csv(OUTPUT / "D4_experiment_log.csv")
display(log)
predictions = pd.read_csv(OUTPUT / "D4_test_predictions.csv")
display(predictions)

## Interpretation

The supplied small quarterly sample favours the persistence benchmark on validation. Candidate algorithms are retained as documented challengers, but no more complex model is selected unless it improves the validation result. The test metrics are an academic backtest, not a production guarantee. Historical publication timestamps are not present in the source workbook, so one-quarter lagging reduces target-quarter leakage but does not fully prove point-in-time availability.